In [ ]:
param_intermediate_location = "/home/jovyan/Cloud Storage/naa-vre-user-data"
conf_tmp_data = "/tmp/data"
conf_data_input_location <- "/home/jovyan/Cloud Storage/naa-vre-public/vl-veluwe-forest-model/"

In [ ]:
# Extract biomass-succession log outputs from all runs
# ====================================================
# Set the path to your outputs directory
output_base_path <- file.path(param_intermediate_location, paste0("workflow_run_", "20260819-0934", "/outputs"))

# Get list of all run folders (001-729)
run_folders <- list.dirs(path = output_base_path, full.names = TRUE, recursive = FALSE)
run_numbers <- basename(run_folders)

total_runs <- length(run_numbers)

# Unzip each output.zip file
for (i in seq_along(run_numbers)) {
  run_num <- run_numbers[i]
  tar_file <- file.path(output_base_path, run_num, "outputs.tar.gz")
  output_dir <- file.path(output_base_path, run_num)
  
  # Check if already unzipped - if there are files besides outputs.zip, skip
  existing_files <- list.files(output_dir)
  if (any(existing_files != "outputs.tar.gz" & existing_files != "Metadata")) {
    cat(sprintf("[%d/%d] Skipping %s - already unzipped\n", i, total_runs, run_num))
    next
  }
  
  # Check if zip file exists
  if (!file.exists(tar_file)) {
    cat(sprintf("[%d/%d] ERROR: %s - zip file not found\n", i, total_runs, run_num))
    next
  }
  
  cat(sprintf("[%d/%d] Unzipping %s...\n", i, total_runs, run_num))
  untar(tar_file, 
        exdir = output_dir)
}

In [ ]:
# Create empty list to store dataframes
all_data <- list()
output_base_path <- file.path(param_intermediate_location, paste0("workflow_run_", "20260819-0934", "/outputs"))

# Loop through each run folder
for (run_num in run_numbers) {

  # Build path to the biomass log file
  biomass_file <- file.path(output_base_path, run_num, "Biomass-succession-log.csv")
  
    # Read the CSV
    data <- readr::read_csv(biomass_file, show_col_types = FALSE)
    
    # Select only the columns you care about and add run column
    data_clean <- data |>
      dplyr::select(Time, AvgLiveB, AvgAG_NPP) |>
      dplyr::mutate(run = run_num)
    
    # Add to list
    all_data[[run_num]] <- data_clean
}

# Combine all dataframes into one
combined_data <- dplyr::bind_rows(all_data)

readr::write_csv(combined_data, paste0(output_base_path, "combined_biomass_succession_data.csv"))